# Deep Error Analysis

Goes beyond "correct/wrong" to understand **why** and **when** vision hurts reasoning.

## Analyses
1. **Disagreement analysis** — What distinguishes problems where text wins vs image wins?
2. **Problem difficulty features** — Steps, numbers, magnitude, operation types
3. **Error propagation** — Where in the reasoning chain do errors first occur?
4. **Correlation analysis** — What problem features predict the modality gap?
5. **Qualitative case studies** — Annotated examples of representative failures

**This notebook analyzes EXISTING results** — run the benchmark first, then use this.

In [ ]:
!pip install scipy pandas matplotlib seaborn --quiet
!git clone https://github.com/Ro-netizen004/vlm-modality-research.git /content/repo 2>/dev/null || echo 'exists'
import sys
sys.path.insert(0, '/content/repo')

from google.colab import drive
drive.mount('/content/drive')
RESULTS_DIR = '/content/drive/MyDrive/vlm_research_results'

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.error_analysis import (
    analyze_disagreements, format_disagreement_report,
    compute_problem_features_df, analyze_error_propagation,
    analyze_modality_gap_correlates,
)
from src.evaluation import extract_numeric_answer

# ── Load results ──
# Change these to match your model
MODEL_SHORT = 'Qwen2-VL-2B-Instruct'
RESULTS_CSV = os.path.join(RESULTS_DIR, MODEL_SHORT, 'gsm8k_results.csv')

df = pd.read_csv(RESULTS_CSV)
print(f'Loaded {len(df)} results for {MODEL_SHORT}')
df.head()

## 1. Disagreement Analysis

In [ ]:
analysis = analyze_disagreements(
    text_correct=df['correct_text'].tolist(),
    img_correct=df['correct_rendered'].tolist(),
    questions=df['question'].tolist(),
    references=df['reference'].tolist(),
)
print(format_disagreement_report(analysis))

In [ ]:
# ── Visualize: What makes vision-hurt vs vision-help problems different? ──

groups = {
    'Text wins': analysis['text_only_correct']['stats'],
    'Image wins': analysis['img_only_correct']['stats'],
    'Both correct': analysis['both_correct']['stats'],
    'Both wrong': analysis['both_wrong']['stats'],
}

metrics = ['avg_steps', 'avg_numbers', 'avg_question_length']
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, metric in zip(axes, metrics):
    labels = [g for g in groups if groups[g]]
    values = [groups[g].get(metric, 0) for g in labels]
    colors = ['#C44E52', '#55A868', '#4C72B0', '#8172B2']
    ax.bar(labels, values, color=colors[:len(labels)], edgecolor='black')
    ax.set_title(metric.replace('avg_', 'Avg ').replace('_', ' ').title())
    ax.grid(axis='y', alpha=0.3)
    for i, v in enumerate(values):
        ax.text(i, v + 0.1, f'{v:.1f}', ha='center', fontsize=10, fontweight='bold')

plt.suptitle(f'Problem Characteristics by Outcome Group — {MODEL_SHORT}', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, MODEL_SHORT, 'disagreement_features.png'), dpi=150)
plt.show()

## 2. Problem Difficulty vs Accuracy

In [ ]:
features = compute_problem_features_df(df['question'].tolist(), df['reference'].tolist())
features['text_correct'] = df['correct_text']
features['img_correct'] = df['correct_rendered']

# Accuracy by number of reasoning steps
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: Accuracy vs steps
step_groups = features.groupby('num_steps').agg(
    text_acc=('text_correct', 'mean'),
    img_acc=('img_correct', 'mean'),
    count=('text_correct', 'count')
).reset_index()
step_groups = step_groups[step_groups['count'] >= 10]  # filter small groups

ax = axes[0]
ax.plot(step_groups['num_steps'], step_groups['text_acc']*100, 'o-', color='#4C72B0', label='Text', linewidth=2)
ax.plot(step_groups['num_steps'], step_groups['img_acc']*100, 's-', color='#55A868', label='Image', linewidth=2)
ax.set_xlabel('Number of Reasoning Steps')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Accuracy vs Problem Complexity')
ax.legend()
ax.grid(alpha=0.3)

# Panel 2: Accuracy vs question length (binned)
features['length_bin'] = pd.cut(features['question_length'], bins=5)
len_groups = features.groupby('length_bin').agg(
    text_acc=('text_correct', 'mean'),
    img_acc=('img_correct', 'mean'),
    count=('text_correct', 'count')
).reset_index()

ax = axes[1]
x = range(len(len_groups))
ax.bar([i-0.2 for i in x], len_groups['text_acc']*100, 0.35, color='#4C72B0', label='Text')
ax.bar([i+0.2 for i in x], len_groups['img_acc']*100, 0.35, color='#55A868', label='Image')
ax.set_xticks(x)
ax.set_xticklabels([str(b) for b in len_groups['length_bin']], fontsize=8, rotation=45)
ax.set_xlabel('Question Length (words)')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Accuracy vs Question Length')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# Panel 3: Modality gap vs answer magnitude
features['modality_gap'] = features['text_correct'].astype(int) - features['img_correct'].astype(int)
valid = features[features['answer_magnitude'].notna()]
valid['log_magnitude'] = np.log10(valid['answer_magnitude'].clip(lower=1))
valid['mag_bin'] = pd.cut(valid['log_magnitude'], bins=5)

mag_groups = valid.groupby('mag_bin')['modality_gap'].mean().reset_index()
ax = axes[2]
ax.bar(range(len(mag_groups)), mag_groups['modality_gap']*100,
       color=['#C44E52' if g>0 else '#55A868' for g in mag_groups['modality_gap']], edgecolor='black')
ax.set_xticks(range(len(mag_groups)))
ax.set_xticklabels([str(b) for b in mag_groups['mag_bin']], fontsize=8, rotation=45)
ax.set_xlabel('log10(Answer Magnitude)')
ax.set_ylabel('Modality Gap (pp)')
ax.set_title('Does Answer Size Affect Modality Gap?')
ax.axhline(y=0, color='black', linewidth=0.5)
ax.grid(axis='y', alpha=0.3)

plt.suptitle(f'Problem Difficulty Analysis — {MODEL_SHORT}', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, MODEL_SHORT, 'difficulty_analysis.png'), dpi=150)
plt.show()

## 3. Error Propagation

In [ ]:
# Analyze where errors first occur in the reasoning chain
wrong_text = df[df['correct_text'] == False]
wrong_img = df[df['correct_rendered'] == False]

text_propagation = [analyze_error_propagation(str(row['pred_text']), str(row['reference']))
                    for _, row in wrong_text.iterrows()]
img_propagation = [analyze_error_propagation(str(row['pred_rendered']), str(row['reference']))
                   for _, row in wrong_img.iterrows()]

# Distribution of error positions
text_positions = [p['error_position_pct'] for p in text_propagation
                  if 'error_position_pct' in p]
img_positions = [p['error_position_pct'] for p in img_propagation
                 if 'error_position_pct' in p]

fig, ax = plt.subplots(figsize=(10, 5))
bins = np.linspace(0, 100, 11)
if text_positions:
    ax.hist(text_positions, bins=bins, alpha=0.6, color='#4C72B0', label=f'Text errors (n={len(text_positions)})', edgecolor='black')
if img_positions:
    ax.hist(img_positions, bins=bins, alpha=0.6, color='#C44E52', label=f'Image errors (n={len(img_positions)})', edgecolor='black')
ax.set_xlabel('Error Position in Reasoning Chain (%)')
ax.set_ylabel('Count')
ax.set_title(f'When Do Errors First Occur? — {MODEL_SHORT}')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, MODEL_SHORT, 'error_propagation.png'), dpi=150)
plt.show()

if text_positions and img_positions:
    print(f'Text errors: mean position = {np.mean(text_positions):.0f}% through chain')
    print(f'Image errors: mean position = {np.mean(img_positions):.0f}% through chain')
    if np.mean(img_positions) < np.mean(text_positions):
        print('→ Image errors occur EARLIER in reasoning — vision disrupts initial parsing')
    else:
        print('→ Image errors occur LATER — vision parses correctly but reasoning drifts')

## 4. Correlation Analysis

In [ ]:
correlations = analyze_modality_gap_correlates(
    df['correct_text'].tolist(), df['correct_rendered'].tolist(),
    df['question'].tolist(), df['reference'].tolist()
)

print('\nCorrelations with modality gap (positive = text benefits more):')
print('-' * 70)
for _, row in correlations.sort_values('spearman_p').iterrows():
    sig = '***' if row['spearman_p']<0.001 else '**' if row['spearman_p']<0.01 else '*' if row['spearman_p']<0.05 else 'ns'
    print(f"  {row['feature']:25s}  r={row['spearman_r']:+.3f}  p={row['spearman_p']:.4f}  {sig}")

correlations.to_csv(os.path.join(RESULTS_DIR, MODEL_SHORT, 'gap_correlations.csv'), index=False)

## 5. Qualitative Case Studies

In [ ]:
# Show representative examples of each disagreement type

def show_case(idx, df):
    row = df.iloc[idx]
    print(f'\nProblem {idx}:')
    print(f'  Q: {row["question"][:150]}...')
    ref_num = extract_numeric_answer(str(row['reference']))
    print(f'  Reference answer: {ref_num}')
    text_num = extract_numeric_answer(str(row['pred_text'])) if 'pred_text' in row else None
    img_num = extract_numeric_answer(str(row['pred_rendered'])) if 'pred_rendered' in row else None
    print(f'  Text prediction:  {text_num}  ({"✓" if row["correct_text"] else "✗"})')
    print(f'  Image prediction: {img_num}  ({"✓" if row["correct_rendered"] else "✗"})')
    if text_num and img_num and ref_num:
        print(f'  Text error: {abs(text_num - ref_num):.0f}  |  Image error: {abs(img_num - ref_num):.0f}')

# Text wins (vision hurt)
text_wins = df[(df['correct_text'] == True) & (df['correct_rendered'] == False)]
print(f'\n{"="*60}')
print(f'TEXT WINS (vision hurt) — {len(text_wins)} cases')
print(f'{"="*60}')
for idx in text_wins.index[:5]:
    show_case(idx, df)

# Image wins (vision helped)
img_wins = df[(df['correct_text'] == False) & (df['correct_rendered'] == True)]
print(f'\n{"="*60}')
print(f'IMAGE WINS (vision helped) — {len(img_wins)} cases')
print(f'{"="*60}')
for idx in img_wins.index[:5]:
    show_case(idx, df)